In [1]:
#!/usr/bin/env python3

import os
import pandas as pd


def main():
    # Caminho fixo para o seu CSV
    csv_path = os.path.join(
        os.getcwd(),
        "results",
        "results.csv"
    )

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Arquivo não encontrado: {csv_path}")

    df = pd.read_csv(csv_path)

    # Filtrar apenas execuções válidas
    df_ok = df[df["status"] == "ok"].copy()

    if df_ok.empty:
        print("Nenhuma linha válida encontrada.")
        return

    # ======= ANÁLISE PRINCIPAL =======
    summary = (
        df_ok.groupby("model", as_index=False)
        .agg(
            mean_abs_error=("abs_error", "mean"),
            median_abs_error=("abs_error", "median"),
            mean_time=("time_per_sample", "mean"),
        )
        .sort_values("mean_abs_error")
    )

    print("\n===== RESULTADOS =====")
    print(summary)

    # Salvar resumo
    summary_path = os.path.splitext(csv_path)[0] + "_summary.csv"
    summary.to_csv(summary_path, index=False)

    print(f"\n✅ Summary salvo em: {summary_path}")
    print(f"Total de linhas analisadas: {len(df_ok)}")


if __name__ == "__main__":
    main()


===== RESULTADOS =====
           model  mean_abs_error  median_abs_error  mean_time
3       EMQ_BCTS        0.048167          0.024204   0.000746
1  DDM_KMeansSup        0.162337          0.093971   0.000004
2        DDM_PCA        0.162337          0.093971   0.000006
0        DDM_GMM        0.162337          0.093971   0.000004

✅ Summary salvo em: /var/new_homes/julio/mestrado/mestrado-dyssyn/experiments/exp_025/results/results_summary.csv
Total de linhas analisadas: 28800


In [2]:
import pandas as pd
import plotly.io as pio
pio.renderers.default = "browser"
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ===============================
# 1. Carregar dados
# ===============================
df = pd.read_csv("results/results.csv")

# Manter apenas execuções válidas
df = df[df["status"] == "ok"].copy()

# ===============================
# 2. Configurações
# ===============================
datasets = df["dataset"].unique()
n_datasets = len(datasets)

fig = make_subplots(
    rows=n_datasets,
    cols=1,
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.02
)

# ===============================
# 3. Plot por dataset
# ===============================
for i, ds in enumerate(datasets, 1):

    df_ds = df[df["dataset"] == ds].copy()

    # Ordenar modelos pela mediana do erro nesse dataset
    ordem_local = (
        df_ds.groupby("model")["abs_error"]
        .median()
        .sort_values()
        .index
        .tolist()
    )

    for model in ordem_local:
        df_model = df_ds[df_ds["model"] == model]

        fig.add_trace(
            go.Box(
                y=df_model["abs_error"],
                name=model,
                boxpoints="outliers",
                legendgroup=model,
                showlegend=(i == 1)
            ),
            row=i,
            col=1
        )

# ===============================
# 4. Layout final
# ===============================
fig.update_layout(
    height=n_datasets * 350,
    template="plotly_white",
    title="Performance Local: Modelos Ordenados do Melhor para o Pior por Dataset",
    margin=dict(t=80, b=40, l=40, r=40),
    showlegend=True
)

fig.update_yaxes(title_text="MAE")

fig.show()

# Opcional: salvar HTML
#fig.write_html("exp_023_boxplots.html")

In [1]:
import pandas as pd
import plotly.express as px


# ===============================
# 1. Carregar dados
# ===============================
df = pd.read_csv("results/results.csv")

# manter apenas execuções válidas
df = df[df["status"] == "ok"].copy()

# ===============================
# 2. Calcular erro médio por dataset e modelo
# ===============================
df_mean = (
    df
    .groupby(["dataset", "model"], as_index=False)
    .agg({"abs_error": "mean"})
)

df_mean = df_mean.rename(columns={"abs_error": "Erro"})

# ===============================
# 3. Plot
# ===============================
fig = px.line(
    df_mean,
    x="dataset",
    y="Erro",
    color="model",
    markers=True,
    title="Erro médio por dataset — exp_023"
)

fig.update_layout(
    xaxis_title="Dataset",
    yaxis_title="Erro absoluto médio (MAE)",
    legend_title="Modelo",
    hovermode="x unified",
    template="plotly_white"
)

fig.update_xaxes(tickangle=45)

# Paleta altamente contrastante
color_map = {
    "cluster_pca": "#000000",      # preto
    "cluster_gmm": "#E69F00",      # laranja forte
    "cluster_kmeans": "#009E73",   # verde forte
    "dys_topsoe": "#D55E00",       # vermelho queimado
}

fig = px.line(
    df_mean,
    x="dataset",
    y="Erro",
    color="model",
    markers=True,
    color_discrete_map=color_map,
    title="Erro médio por dataset — exp_023"
)

# ===============================
# 4. Salvar HTML
# ===============================
#fig.write_html("exp_023_erro_medio_por_dataset.html")


fig.show()